In [1]:
# Create customer master data and transaction records
import pandas as pd
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "segment": ["new", "returning", "new", "business"],
    "region": ["North", "South", "West", "North"],
})

transactions = pd.DataFrame({
    "transaction_id": [1, 2, 3, 4, 5],
    "customer_id": ["C01", "C01", "C02", "C03", "C05"],
    "amount": [50, 75, 120, 40, 200],
})


In [12]:
# Keep every transaction while adding matching customer attributes
# left join from transactions to customers.
transactions_with_customers=transactions.merge(
    customers,
    on="customer_id",
    how="left")
transactions_with_customers

,transaction_id,customer_id,amount,segment,region
0,1,C01,50,new,North
1,2,C01,75,new,North
2,3,C02,120,returning,South
3,4,C03,40,new,West
4,5,C05,200,NaN,NaN


In [9]:
# Find transactions whose customer ID was not found in the master table
missing_customer_transaction=transactions_with_customers[transactions_with_customers.isna().any(axis=1)]["transaction_id"].tolist()
print("Transactions with missing customer information: ",missing_customer_transaction)

Transcations with missing customer information:  [5]


In [13]:
# Aggregate known transaction revenue by customer segment
revenue_by_segment=(
    transactions_with_customers.groupby("segment",as_index=False)["amount"]
    .sum()
    .rename(columns={"amount":"revenue"})
    )
revenue_by_segment

,segment,revenue
0,new,165
1,returning,120


In [21]:
# Check key uniqueness and repeated customer transactions
print("Customer table has unique customer IDs: ",customers["customer_id"].is_unique)
print("Customer with multiple transactions: ",transactions_with_customers.loc[transactions_with_customers["customer_id"].duplicated(),"customer_id"].unique().tolist())

Customer table has unique customer IDs:  True
Customer with multiple transactions:  ['C01']


### Important notes

- A left join keeps every row from the left table; unmatched customer fields become missing values.
- A one-to-many relationship is expected for transactions, but customer master keys should usually be unique.
- Grouping by `segment` excludes missing segments by default, so unmatched transactions are not included in the segment totals.
- Use `validate="many_to_one"` when the right table must contain one record per customer.
- Check row counts and key uniqueness after merges to detect unexpected row multiplication.
